# 04 - Baseline models

In [ ]:
import sys
sys.path.append("..")

import numpy as np

from experiments.data import load_data
from experiments.evaluation import select_candidate, record_result, diagnostics, comparison_table, inverse_log
from experiments.estimators import structured_preprocessor, MedianPricePerSqm

In [ ]:
parts, manifest = load_data("data/processed/model_input.parquet")
train, val, test = parts["train"], parts["validation"], parts["test"]

y_train, y_val = train.price.to_numpy(), val.price.to_numpy()
log_y_train, log_y_val = np.log1p(y_train), np.log1p(y_val)

manifest["split_counts"]

## Helper

`run` fits each set of parameters, keeps the one with the lowest validation MAE, then scores it on the test set and saves the results to `reports/experiments/`.

In [ ]:
FORMAT = {"MAE": "{:,.0f}", "RMSE": "{:,.0f}", "R2": "{:.4f}", "MAPE": "{:.2f}"}


def run(key, candidates, fit, predict, X_val, X_test, extra=None):
    model, params, tuning = select_candidate(candidates, fit, y_val, lambda m: predict(m, X_val))
    display(tuning)
    print("Best params:", params)

    val_pred = predict(model, X_val)
    test_pred = predict(model, X_test)
    _, scores = record_result(key, manifest, parts, val_pred, test_pred, params, tuning, extra=extra)
    display(scores.style.format(FORMAT))
    diagnostics(key, test.price, test_pred)
    return model

## Dummy Regressor

Always predicts the mean training price.

In [ ]:
from sklearn.dummy import DummyRegressor


def fit_dummy(params):
    return DummyRegressor(**params).fit(np.zeros((len(train), 1)), y_train)


def predict_dummy(model, X):
    return model.predict(np.zeros((len(X), 1)))


dummy = run("dummy", [{"strategy": "mean"}], fit_dummy, predict_dummy, val, test)

## Median price per m²

Median price per m² for each (province, district, property type), multiplied by the area. If a group is not in the training data, it falls back to (province, property type), then property type, then the overall median.

In [ ]:
def fit_median(params):
    return MedianPricePerSqm().fit(train, y_train)


def predict_median(model, X):
    return model.predict(X)


median = run("median", [{}], fit_median, predict_median, val, test)

## Ridge Regression

One-hot encoding and scaling, trained on `log1p(price)`.

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import Ridge


def fit_ridge(params):
    model = make_pipeline(structured_preprocessor(scale=True), Ridge(**params, solver="lsqr", tol=1e-5))
    return model.fit(train, log_y_train)


def predict_ridge(model, X):
    return inverse_log(model.predict(X), y_train)


ridge_params = [{"alpha": 1.0}, {"alpha": 10.0}, {"alpha": 100.0}]
ridge = run("ridge", ridge_params, fit_ridge, predict_ridge, val, test)

## Compare models

In [ ]:
comparison_table(manifest).style.format(FORMAT)